In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import random
import shutil
from pathlib import Path

# Input dataset path
input_dir = Path("/kaggle/input/datasets/sathvik2006/sample-35k")

# Output directory
output_dir = Path("/kaggle/working")

train_dir = output_dir / "train"
val_dir = output_dir / "val"
test_dir = output_dir / "test"

# Create output folders
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

# Collect all files
all_files = []
for root, dirs, files in os.walk(input_dir):
    for file in files:
        all_files.append(os.path.join(root, file))

# Shuffle files
random.shuffle(all_files)

# Split sizes
total = len(all_files)
train_split = int(0.7 * total)
val_split = int(0.85 * total)

train_files = all_files[:train_split]
val_files = all_files[train_split:val_split]
test_files = all_files[val_split:]

print("Total files:", total)
print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

# Copy files
for file in train_files:
    shutil.copy(file, train_dir)

for file in val_files:
    shutil.copy(file, val_dir)

for file in test_files:
    shutil.copy(file, test_dir)

print("Dataset successfully split!")

In [ ]:
!rm -rf /kaggle/working/train
!rm -rf /kaggle/working/val
!rm -rf /kaggle/working/test

In [ ]:
import os
import random
import shutil
from pathlib import Path

dataset_path = Path("/kaggle/input/datasets/sathvik2006/sample-35k/sample_35k")

images_dir = dataset_path / "images"
annos_dir = dataset_path / "annos"

output_dir = Path("/kaggle/working")

train_img = output_dir / "train/images"
train_ann = output_dir / "train/annos"

val_img = output_dir / "val/images"
val_ann = output_dir / "val/annos"

test_img = output_dir / "test/images"
test_ann = output_dir / "test/annos"

for p in [train_img, train_ann, val_img, val_ann, test_img, test_ann]:
    p.mkdir(parents=True, exist_ok=True)

anno_files = [f for f in os.listdir(annos_dir) if f.endswith(".json")]

random.shuffle(anno_files)

total = len(anno_files)

train_split = int(0.7 * total)
val_split = int(0.85 * total)

train_files = anno_files[:train_split]
val_files = anno_files[train_split:val_split]
test_files = anno_files[val_split:]

print("Total:", total)
print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))


def copy_pairs(file_list, img_out, ann_out):
    for ann in file_list:

        img = ann.replace(".json", ".jpg")

        src_ann = annos_dir / ann
        src_img = images_dir / img

        if src_img.exists():
            shutil.copy(src_ann, ann_out)
            shutil.copy(src_img, img_out)


copy_pairs(train_files, train_img, train_ann)
copy_pairs(val_files, val_img, val_ann)
copy_pairs(test_files, test_img, test_ann)

print("Dataset split correctly!")

In [ ]:
import json
import os
import pandas as pd
from tqdm import tqdm

train_ann_dir = "/kaggle/working/train/annos"

data = []

for file in tqdm(os.listdir(train_ann_dir)):

    path = os.path.join(train_ann_dir, file)

    with open(path) as f:
        ann = json.load(f)

    for key, obj in ann.items():

        # Only process clothing objects
        if isinstance(obj, dict) and "category_name" in obj:
            data.append(obj["category_name"])

df = pd.DataFrame(data, columns=["category"])

class_counts = df["category"].value_counts()

print(class_counts)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = df["category"].unique()

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df["category"]
)

class_weights = {c: float(w) for c, w in zip(classes, weights)}

print(class_weights)

In [ ]:
import json
import os
import pandas as pd
from tqdm import tqdm

train_ann_dir = "/kaggle/working/train/annos"

records = []

for file in tqdm(os.listdir(train_ann_dir)):

    path = os.path.join(train_ann_dir, file)
    image_name = file.replace(".json", ".jpg")

    with open(path) as f:
        ann = json.load(f)

    for key, obj in ann.items():

        if isinstance(obj, dict) and "category_name" in obj:

            category = obj["category_name"]
            bbox = obj["bounding_box"]
            segmentation = obj["segmentation"]

            records.append({
                "image": image_name,
                "category": category,
                "bbox": bbox,
                "segmentation": segmentation
            })

df_annotations = pd.DataFrame(records)

print("Total objects:", len(df_annotations))
df_annotations.head()

In [ ]:
df_annotations.to_csv("/kaggle/working/train_annotations.csv", index=False)

print("Saved parsed annotation dataset")

In [1]:
def parse_annotations(ann_dir, save_path):

    records = []

    for file in tqdm(os.listdir(ann_dir)):

        path = os.path.join(ann_dir, file)
        image_name = file.replace(".json", ".jpg")

        with open(path) as f:
            ann = json.load(f)

        for key, obj in ann.items():

            if isinstance(obj, dict) and "category_name" in obj:

                records.append({
                    "image": image_name,
                    "category": obj["category_name"],
                    "bbox": obj["bounding_box"],
                    "segmentation": obj["segmentation"]
                })

    df = pd.DataFrame(records)
    df.to_csv(save_path, index=False)

    print("Saved:", save_path)


parse_annotations("/kaggle/working/train/annos", "/kaggle/working/train_annotations.csv")
parse_annotations("/kaggle/working/val/annos", "/kaggle/working/val_annotations.csv")
parse_annotations("/kaggle/working/test/annos", "/kaggle/working/test_annotations.csv")

NameError: name 'tqdm' is not defined